# 1. Importando bibliotecas

In [0]:
import requests 
import pandas as pd
from datetime import datetime

# 2. Extração e Transformação dos dados

In [0]:
def extrair_dados_bitcoin():
    url = 'https://api.coinbase.com/v2/prices/spot' #API da Coinbase para obter o preço atual do Bitcoin
    response = requests.get(url)
    return response.json()

In [0]:
dados_bitcoin = extrair_dados_bitcoin()
print(dados_bitcoin)

In [0]:
def extrair_cotacao_usd_brl():
    api_key = '2291516f22814424a5eca3af2106d9e0'
    url = f'https://api.currencyfreaks.com/v2.0/rates/latest?apikey={api_key}'
    response = requests.get(url)
    return response.json()

In [0]:
dados_cotacao = extrair_cotacao_usd_brl()
print(dados_cotacao['rates']['BRL'])

In [0]:
def tratar_dados_bitcoin(dados_json, taxa_usd_brl):
    valor_usd = float(dados_json['data']['amount'])
    criptomoeda = dados_json['data']['base']
    moeda_original = dados_json['data']['currency']

    #convertendo de USD para BRL
    valor_brl = valor_usd * taxa_usd_brl

    #adicoinando timesptamp
    timestamp = datetime.now()

    dado_tratado = [{
        "valor_usd": valor_usd,
        "valor_brl": valor_brl,
        "criptomoeda": criptomoeda,
        "moeda_original": moeda_original,
        "taxa_conversao_usd_brl": taxa_usd_brl,
        "timestamp": timestamp
    }]

    return dado_tratado

In [0]:
#extraindo os dados
dados_bitcoin = extrair_dados_bitcoin()
dados_cotacao = extrair_cotacao_usd_brl()

#extraindo a taxa de conversão de UDS para BRL
taxa_usd_brl = float(dados_cotacao['rates']['BRL'])

#tratando os dados e convertendo para BRL
dados_bitcoin_tratado = tratar_dados_bitcoin(dados_bitcoin, taxa_usd_brl)

print(dados_bitcoin_tratado)

# 3. Configurando Unity Catalog

In [0]:
%sql
create catalog if not exists pipe_api_bitcoin
comment "Catálogo de demonstração do workshop Pipeline de dados API Bitcoin"

In [0]:
%sql
create schema if not exists pipe_api_bitcoin.lakehouse
comment "Schema Lakehouse para salvar dados processados";

In [0]:
%sql
create volume if not exists pipe_api_bitcoin.lakehouse.raw_files
comment "Volume para arquivos brutos de ingestão inicial";

# 4. Criando Pandas DF

# 05. Salvando em Json usando PySpark